In [1]:
import os
print(os.listdir("/kaggle/input/datasets/susandilinn/ait-lectures"))

['Artificial Intelligence Technology_lecture 1.pdf', 'Artificial Intelligence Technology_lecture 6.pdf', 'Artificial Intelligence Technology_lecture 2.pdf', 'Artificial Intelligence Technology_lecture 3.pdf', 'Artificial-Intelligence-Technology_lecture-4.pdf', 'Artificial Intelligence Technology_lecture 7.pdf', 'Artificial Intelligence Technology_lecture 8.pdf', 'Artificial Intelligence Technology_lecture 5.pdf', 'Artificial Intelligence Technology_lecture 9.pdf']


In [2]:
# INSTALLS
!pip install pdfplumber transformers scikit-learn joblib sentencepiece -q

import os
import re
import glob
import torch
import pdfplumber
import joblib
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans


# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# LOAD MODELS
bert_path = "/kaggle/input/datasets/susandilinn/models/my_bert_model/my_bert_model"

tokenizer = AutoTokenizer.from_pretrained(bert_path)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_path)

bert_model.to(device)
bert_model.eval()

logistic_model = joblib.load("/kaggle/input/datasets/susandilinn/models/logistic_model.pkl")
tfidf_vectorizer = joblib.load("/kaggle/input/datasets/susandilinn/models/tfidf_vectorizer.pkl")


# PDF EXTRACTION
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# SENTENCE SPLITTING (IMPROVED)
def split_into_sentences(text):

    text = text.replace("\n", " ")

    raw = re.split(r'[.!?•]\s+|\s{2,}', text)

    sentences = []

    for s in raw:

        s = s.strip()

        if len(s.split()) < 8:
            continue

        sentences.append(s)

    return sentences


# REMOVE REPEATED WORDS
def remove_repeated_words(sentences):

    cleaned = []

    for s in sentences:

        words = s.split()

        unique = []

        for w in words:

            if len(unique) == 0 or w != unique[-1]:
                unique.append(w)

        cleaned.append(" ".join(unique))

    return cleaned


# REMOVE SLIDE NOISE
def remove_slide_noise(sentences):

    cleaned = []

    patterns = [
        r"lecture\s*\d+",
        r"slide\s*\d+",
        r"cpts\s*\d+",
        r"page\s*\d+",
        r"\b\d+\s*/\s*\d+\b",
        r"\bfigure\b",
        r"\btable\b"
    ]

    for s in sentences:

        s_lower = s.lower()

        if any(re.search(p, s_lower) for p in patterns):
            continue

        if s.isupper() and len(s.split()) < 6:
            continue

        cleaned.append(s)

    return cleaned


# REMOVE URLS
def remove_urls(sentences):

    cleaned = []

    for s in sentences:

        s = re.sub(r"http\S+", "", s)
        s = re.sub(r"www\.\S+", "", s)

        cleaned.append(s.strip())

    return cleaned


# REMOVE SYMBOL HEAVY SENTENCES
def remove_symbols(sentences):

    cleaned = []

    for s in sentences:

        non_letters = len(re.findall(r"[^a-zA-Z\s]", s))

        if non_letters / max(1, len(s)) > 0.25:
            continue

        cleaned.append(s)

    return cleaned


# LOGISTIC IMPORTANCE FILTER
def logistic_importance_filter(sentences, keep_ratio=0.7):

    if len(sentences) <= 5:
        return sentences

    vectors = tfidf_vectorizer.transform(sentences)

    probs = logistic_model.predict_proba(vectors)

    scores = probs[:,1]

    ranked = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)

    top_k = max(5, int(len(ranked) * keep_ratio))

    return [s[0] for s in ranked[:top_k]]


# BERT SCORING
def bert_score_sentences(sentences, batch_size=16):

    scores = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():

            outputs = bert_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

        for j, sentence in enumerate(batch):

            scores.append((sentence, probs[j][1].item()))

    return scores


# SELECT TOP SENTENCES
def select_top_sentences(scored_sentences, ratio=0.45):

    ranked = sorted(scored_sentences, key=lambda x: x[1], reverse=True)

    top_k = max(1, int(len(ranked) * ratio))

    selected = ranked[:top_k]

    selected_sentences = [s[0] for s in selected]

    original_order = [s[0] for s in scored_sentences]

    selected_sentences.sort(key=lambda x: original_order.index(x))

    return selected_sentences


# REDUNDANCY REMOVAL
def remove_redundancy(sentences, threshold=0.8):

    if len(sentences) <= 1:
        return sentences

    vectors = tfidf_vectorizer.transform(sentences)

    similarity = cosine_similarity(vectors)

    keep = []

    for i, sent in enumerate(sentences):

        redundant = False

        for j in keep:

            if similarity[i][j] > threshold:
                redundant = True
                break

        if not redundant:
            keep.append(i)

    return [sentences[i] for i in keep]


# TITLE GENERATION
def generate_titles(sentences, num_sections=2):

    from sklearn.feature_extraction.text import TfidfVectorizer

    if len(sentences) == 0:
        return []

    num_sections = min(num_sections, len(sentences))

    if num_sections == 1:
        return [("KEY IDEA", sentences)]

    vectorizer = TfidfVectorizer(stop_words="english")

    X = vectorizer.fit_transform(sentences)

    kmeans = KMeans(n_clusters=num_sections, random_state=42, n_init=10)

    labels = kmeans.fit_predict(X)

    feature_names = vectorizer.get_feature_names_out()

    sections = {}

    for i, label in enumerate(labels):
        sections.setdefault(label, []).append(sentences[i])

    results = []

    for label, sents in sections.items():

        cluster_vec = vectorizer.transform(sents).mean(axis=0)

        top_idx = cluster_vec.A1.argsort()[-3:][::-1]

        words = [feature_names[i] for i in top_idx]

        title = " ".join(words).replace("_", " ").upper()

        results.append((title, sents))

    return results


# HYBRID SUMMARIZER
def summarize_lecture(pdf_path):

    text = extract_text_from_pdf(pdf_path)

    sentences = split_into_sentences(text)

    sentences = remove_repeated_words(sentences)
    sentences = remove_slide_noise(sentences)
    sentences = remove_urls(sentences)
    sentences = remove_symbols(sentences)

    if len(sentences) == 0:
        return None

    # Logistic filter
    filtered = logistic_importance_filter(sentences)

    # BERT scoring
    scored = bert_score_sentences(filtered)

    # Sentence selection
    selected = select_top_sentences(scored)

    # Remove redundancy
    final_sentences = remove_redundancy(selected)

    # Generate titles
    sections = generate_titles(final_sentences)

    summary = ""

    for title, sents in sections:

        summary += f"\nTITLE: {title}\n"

        for s in sents:
            summary += f"- {s}\n"

    return summary


# PROCESS ALL PDFs
output_file_path = "/kaggle/working/all_summaries.txt"

with open(output_file_path, "w", encoding="utf-8") as f:

    for pdf_path in glob.glob("/kaggle/input/datasets/susandilinn/ait-lectures/*.pdf"):

        summary = summarize_lecture(pdf_path)

        if summary:

            name = os.path.basename(pdf_path)

            print("\nFILE:", name)
            print(summary)

            f.write(f"\nFILE: {name}\n")
            f.write(summary)
            f.write("\n")


print("Cheatsheet TXT file created!")
print("Saved at:", output_file_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.0 MB/s eta 0:00:00


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


FILE: Artificial Intelligence Technology_lecture 1.pdf

TITLE: INTELLIGENCE ARTIFICIAL AI
- Narrow AI is a specific type of Artificial Intelligence technology that will enable computers to outperform humans in some very narrowly defined task, unlike general intelligence – narrow intelligence focuses on a single subset of abilities and looks to make strides in that spectrum
- Artificial Super Intelligence (ASI) Artificial Super Intelligence (ASI) is a super intelligent computer that can possess an intelligence that far surpasses that of the brightest and most gifted human minds
- Artificial General intelligence is the capability of a machine to perform the same intellectual tasks as a human to the same standard as humans
- Artificial Super Intelligence (ASI) o Hypothetical AI Artificial Narrow Intelligence (ANI) Artificial Narrow Intelligence (ANI), also called “weak AI”
- Artificial Intelligence is the development of computer systems that are capable of performing tasks that normally 